In [1]:
import warnings

warnings.simplefilter("ignore")
import numpy as np

np.seterr(all="ignore")

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

In [2]:
import matplotlib.pyplot as plt

import astropy.units as u
from threeML import *
from threeML.io.package_data import get_path_of_data_file

In [3]:
spectrum = Powerlaw()
source = PointSource("crab", ra=83.63, dec=22.01, spectral_shape=spectrum)
spectrum.piv = 1 * u.TeV
spectrum.piv.fix = True
spectrum.piv.unit = u.TeV

spectrum.K = 3.37e-11 / (u.TeV * u.cm**2 * u.s)  # norm (in 1/(TeV cm2 s))
spectrum.K.unit = 1 / (u.TeV * u.cm**2 * u.s)
#spectrum.K.min_value = 1e-12 / (u.TeV * u.cm**2 * u.s)
#spectrum.K.max_value = 1e-10 / (u.TeV * u.cm**2 * u.s)

spectrum.index = -2.53
#spectrum.index.min_value = -4

#spectrum.index.max_value = -1



In [4]:
likelihood_model = Model(source)


In [5]:
likelihood_model.display()

Model summary:
==============

                  N
Point sources     1
Extended sources  0
Particle sources  0

Free parameters (2):
--------------------

                                  value min_value max_value            unit
crab.spectrum.main.Powerlaw.K       0.0       0.0    1000.0  TeV-1 s-1 cm-2
crab.spectrum.main.Powerlaw.index -2.53     -10.0      10.0                

Fixed parameters (3):
(abridged. Use complete=True to see all fixed parameters)


Properties (0):
--------------------

(none)


Linked parameters (0):
----------------------

(none)

Independent variables:
----------------------

(none)

Linked functions (0):
----------------------

(none)

In [6]:
from GammapyLike import GammapyLike

ImportError: attempted relative import with no known parent package

In [ ]:
VERITAS = GammapyLike("VERITAS", config_file="config.yaml")

In [ ]:
data = DataList(VERITAS)

In [ ]:
jl = JointLikelihood(likelihood_model, data, verbose=True)

In [ ]:
jl.set_minimizer("minuit")

res = jl.fit()



In [ ]:
grid_minimizer = GlobalMinimization("grid")

# Create an instance of a local minimizer, which will be used by GRID
local_minimizer = LocalMinimization("minuit")

# Define a grid for mu as 10 steps between 2 and 80
my_grid = {likelihood_model.crab.spectrum.main.Powerlaw.K: np.logspace(np.log10(1e-12), np.log10(1e-10), 5), 
          likelihood_model.crab.spectrum.main.Powerlaw.index: np.linspace(-3,-1,5)}


grid_minimizer.setup(
    second_minimization=local_minimizer, grid=my_grid
)

jl.set_minimizer(grid_minimizer)


res = jl.fit()